# 10 · Limpieza: Estados oportunidades ganadas por KAM_anonimizado

Reporte de Zoho CRM (13 de noviembre de 2026) con las oportunidades comerciales **ganadas** por cada KAM. Una fila = una oportunidad ganada (una empresa).

- Entrada: `datos/originales/Estados oportunidades ganadas por KAM_anonimizado.xlsx` (hoja `Sheet0`)
- Salida: `datos/limpios/10_oportunidades_ganadas_limpio.csv`

La columna `Importe` viene vacía por la anonimización y no se usa.

In [ ]:
from pathlib import Path
import re

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Cómo viene un reporte agrupado de Zoho
Es el mismo formato del cuaderno 07_08. Las filas 0 a 5 son títulos (con `Cantidad de registros : n`) y la fila 6 tiene los encabezados.
Los datos vienen **agrupados**: cada etiqueta de grupo aparece solo en la primera fila del grupo, con su conteo entre paréntesis (`KAM C ( 10 )`), y las filas siguientes la dejan vacía.

**Cómo se limpia:** se rellena cada celda vacía con el valor de la fila de arriba (`ffill`), se dejan solo las filas que tienen empresa y se quitan los `( n )`.
Al final se comprueba que haya tantos registros como dice el reporte y que cada grupo tenga los que dice su `( n )`.

In [ ]:
def leer_reporte_zoho(archivo, encabezados, columnas_grupo):
    crudo = pd.read_excel(ORIGINALES / archivo, header=None)
    esperados = int(re.search(r"Cantidad de registros : (\d+)", crudo.iloc[4, 0]).group(1))
    assert crudo.iloc[6].tolist() == encabezados, crudo.iloc[6].tolist()  # el reporte no cambió de forma

    datos = crudo.iloc[7:].copy()
    datos = datos[datos[0] != "Grand Total"]
    datos[columnas_grupo] = datos[columnas_grupo].ffill()  # cada fila hereda su grupo
    return datos, esperados


def separar_conteo(serie):
    """'KAM C ( 10 )' -> ('KAM C', 10)"""
    partes = serie.str.extract(r"^(.*?)\s*\(\s*(\d+)\s*\)\s*$")
    assert partes.notna().all().all(), serie[partes.isna().any(axis=1)]
    return partes[0].str.strip(), partes[1].astype(int)


def comprobar_conteos(df, niveles):
    """niveles: [(columna_grupo, columna_conteo), ...] del más general al más específico.
    El número entre paréntesis de cada grupo debe coincidir con las filas que quedaron en él."""
    for k, (_, conteo) in enumerate(niveles):
        grupos = df.groupby([col for col, _ in niveles[:k + 1]])
        assert (grupos.size() == grupos[conteo].first()).all(), (grupos.size(), grupos[conteo].first())

## 2. Oportunidades ganadas
Un solo nivel de grupo: el KAM. `Sin asignar (cuenta ACTHO)` se deja como `Sin asignar`, para que en Looker aparezca como una opción más junto a `KAM A`…`KAM F`.

In [ ]:
ENCABEZADOS = ["Propietario de Trato [anonimizado]", "Nombre de Trato [anonimizado]", "Importe [eliminado]"]
op, esperados = leer_reporte_zoho("Estados oportunidades ganadas por KAM_anonimizado.xlsx", ENCABEZADOS, columnas_grupo=[0])
op.columns = ["kam", "id_empresa", "importe"]
op = op[op["id_empresa"].notna()].drop(columns="importe")

op["kam"], op["n_kam"] = separar_conteo(op["kam"])
op["kam"] = op["kam"].replace({"Sin asignar (cuenta ACTHO)": "Sin asignar"})
op["id_empresa"] = op["id_empresa"].str.strip()
op = op.reset_index(drop=True)
op.groupby("kam").size()

## 3. Validaciones
- Hay tantos registros como dice el reporte (39) y cada KAM tiene los que dice su `( n )`.
- Cada empresa aparece una sola vez y con formato `EMP-0000`.

In [ ]:
assert len(op) == esperados == 39, (len(op), esperados)
comprobar_conteos(op, [("kam", "n_kam")])
assert op["id_empresa"].is_unique and op["id_empresa"].str.fullmatch(r"EMP-\d{4}").all()
assert op["kam"].str.fullmatch(r"KAM [A-Z]|Sin asignar").all()
op = op.drop(columns="n_kam")

## 4. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes. Comparte `kam` e `id_empresa` con el 07_08, el 09 y el 11.

In [ ]:
salida = LIMPIOS / "10_oportunidades_ganadas_limpio.csv"
op.to_csv(salida, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (39, 2)
salida.name